# IBKR Options — Parquet Explorer

Quick notebook to load and inspect every Parquet dataset written by the pipeline into `data/`.

In [ ]:
import os
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# Root of the repo — works regardless of where the notebook is run from
REPO_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DATA_DIR = REPO_ROOT / "data"

DATASETS = [
    "option_chain",
    "spreads",
    "iron_condors",
    "iron_butterflies",
    "calendars",
    "strangles",
    "expected_move",
    "max_pain",
]

def load(name: str) -> pd.DataFrame | None:
    path = DATA_DIR / name
    if not path.exists():
        print(f"⚠️  {name}: not found at {path}")
        return None
    df = pd.read_parquet(path)
    print(f"✅  {name}: {len(df):,} rows × {len(df.columns)} cols")
    return df

print(f"DATA_DIR = {DATA_DIR.resolve()}\n")
dfs = {name: load(name) for name in DATASETS}

## Option Chain

In [ ]:
# ── Bid/Ask/Close quality diagnostic ────────────────────────────────────────
# IBKR returns -1 for bid/ask when the market is closed (documented behaviour:
# https://interactivebrokers.github.io/tws-api/md_receive.html).
# The pipeline converts -1 → None via _safe_float, so 'mid' is derived from
# 'close' as a fallback.  Bid/ask will populate correctly during market hours.
df = dfs["option_chain"]
if df is not None:
    total = len(df)
    def pct(n): return f"{n:,}  ({n/total*100:.1f}%)"

    null_bid   = df["bid"].isna().sum()
    null_ask   = df["ask"].isna().sum()
    null_mid   = df["mid"].isna().sum()
    null_delta = df["delta"].isna().sum()

    has_close = "close" in df.columns
    null_close = df["close"].isna().sum() if has_close else total

    print(f"Total contracts : {total:,}")
    print(f"  bid    null   : {pct(null_bid)}")
    print(f"  ask    null   : {pct(null_ask)}")
    if has_close:
        print(f"  close  null   : {pct(null_close)}")
    else:
        print("  close         : ⚠️  column missing — re-run pipeline to pick up schema change")
    print(f"  mid    null   : {pct(null_mid)}")
    print(f"  delta  null   : {pct(null_delta)}")
    print()

    if null_bid / total > 0.5:
        print("⚠️  Most bid/ask values are null — data was likely collected outside")
        print("   US market hours (9:30–16:00 ET). IBKR returns -1 (→ None) for")
        print("   bid/ask when no live/delayed quote is available.")
        if has_close and null_close / total < 0.5:
            print(f"   ✅ 'close' is populated for {total - null_close:,}/{total:,} contracts")
            print("      and is used as the 'mid' fallback outside market hours.")
    else:
        print("✅  Bid/ask data looks healthy.")

In [ ]:
df = dfs["option_chain"]
if df is not None:
    display(df.head(10))
    print("\n--- dtypes ---")
    display(df.dtypes.to_frame("dtype"))
    print("\n--- describe ---")
    display(df.describe())

## Credit Spreads &amp; Debit Spreads

In [ ]:
df = dfs["spreads"]
if df is not None:
    print("Strategy counts:")
    display(df["strategy"].value_counts())
    print("\n--- Credit Spreads (top 10 by spread_ratio) ---")
    display(
        df[df["strategy"].str.contains("Credit")]
        .sort_values("spread_ratio", ascending=False)
        .head(10)
    )
    print("\n--- Debit Spreads (top 10 by spread_ratio) ---")
    display(
        df[df["strategy"].str.contains("Debit")]
        .sort_values("spread_ratio", ascending=False)
        .head(10)
    )

## Iron Condors

In [ ]:
df = dfs["iron_condors"]
if df is not None:
    display(df.sort_values("spread_ratio", ascending=False).head(10))

## Iron Butterflies

In [ ]:
df = dfs["iron_butterflies"]
if df is not None:
    display(df.sort_values("spread_ratio", ascending=False).head(10))

## Strangles

In [ ]:
df = dfs["strangles"]
if df is not None:
    display(df.sort_values("total_premium", ascending=False).head(10))

## Calendar Spreads

In [ ]:
df = dfs["calendars"]
if df is not None:
    display(df.sort_values("theta_differential", ascending=False).head(10))

## Analytics — Expected Move &amp; Max Pain

In [ ]:
df_em = dfs["expected_move"]
if df_em is not None:
    print("--- Expected Move ---")
    display(df_em)

df_mp = dfs["max_pain"]
if df_mp is not None:
    print("--- Max Pain ---")
    display(df_mp)

## Missing Values &amp; Data Quality

In [ ]:
for name, df in dfs.items():
    if df is None:
        continue
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"✅  {name}: no nulls")
    else:
        print(f"⚠️  {name}:")
        display(nulls.to_frame("null_count"))